In [37]:
!pip install pytorch_lightning

In [38]:
import random, os, pickle, time
import pandas as pd
import numpy as np
import networkx as nx

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

from collections import defaultdict


# Set environment variables for reproducibility and safety
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import precision_score, recall_score, accuracy_score

# 1. Configuration & Seeding
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [39]:
NAME = 'book'
N_CLUSTERS = 15

#### Config

In [40]:
class Config:
    # Đường dẫn dữ liệu
    TCKG_PATH = f"./data/{NAME}/{NAME}_TCKG.csv"  # File CSV chứa đồ thị (như bạn đã gửi)
    # TRAIN_USERS_PATH = "data/train_users.csv" # File chứa danh sách user ID dùng để train
    SAVE_DIR = "./checkpoints"

    ITEM_START_ID = 2891  # book: 2891
    ITEM_END_ID = 11584     # book: 11584

    # Siêu tham số Model
    EMBED_DIM = 64
    HIDDEN_DIM = 256
    HISTORY_LEN = 1   # k' (độ dài lịch sử)
    MAX_PATH_LEN = 3  # K (số bước đi)

    # Siêu tham số Training
    BATCH_SIZE = 32 # Batch lớn giúp RL ổn định hơn
    NUM_EPOCHS = 500
    LEARNING_RATE = 1e-3

    # Thiết bị
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # IDs của các Relation Interaction (Quan trọng cho Reward)
    # Ví dụ: 20=interacted_0, 21=interacted_1, 22=interacted_2
    INTERACTION_CLUSTER_IDS = [21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]



#### TCKG

In [41]:
class TCKG:
    def __init__(self, tckg_csv_path):
        self.adj_list = defaultdict(list)

        print(f"Loading TCKG from {tckg_csv_path}...")
        df_tckg = pd.read_csv(tckg_csv_path, usecols=['head_id', 'relation_id', 'tail_id'])

        offset = int(df_tckg['relation_id'].max())

        data = df_tckg[['head_id', 'relation_id', 'tail_id']].to_numpy() # Using numpy to speedup
        for h, r, t in data:
            h, r, t = int(h), int(r), int(t)

            self.adj_list[h].append((r, t))
            self.adj_list[t].append((r + offset, h)) # Ex: r=5 (watched) -> r_inv=105 (watched_by)

        print(f"TCKG Loaded successfully. Graph construction complete.")

    def get_neighbors(self, node_id):
        return self.adj_list[node_id]

    def get_all_nodes(self):
        return list(self.adj_list.keys())
        rels = self.rel_matrix[node_ids_tensor]
        tails = self.tail_matrix[node_ids_tensor]
        masks = self.mask_matrix[node_ids_tensor]
        return rels, tails, masks

#### TimeAwareRewardFunction

In [42]:
class TimeAwareRewardFunction(nn.Module):
    def __init__(self, user_embs, entity_embs, relation_embs, interaction_cluster_ids, bias_embs=None, temperature= None):
        """
        Args:
            user_embs (nn.Embedding): Embedding của User (e_u)self.max_neighbors = max(len(edges) for edges in self.adj_list.values())
            entity_embs (nn.Embedding): Embedding của Item/Entity (e_v)
            relation_embs (nn.Embedding): Embedding của Relation (dùng để lấy V_U)
            interaction_cluster_ids (list or tensor): Danh sách các Relation ID đại diện cho Time Clusters.
                                                      Ví dụ: [20, 21, 22] ứng với interacted_0, interacted_1...
                                                      Đây chính là tập {V_U^1, ..., V_U^L}
            bias_embs (nn.Embedding, optional): Bias của entity (b_v). Nếu None sẽ tự khởi tạo.
        """
        super(TimeAwareRewardFunction, self).__init__()

        self.user_embs = user_embs
        self.entity_embs = entity_embs
        self.relation_embs = relation_embs

        # Danh sách ID của các cluster tương tác theo thời gian (V_U)
        # Chuyển thành tensor để tính toán song song
        self.register_buffer('cluster_ids', torch.tensor(interaction_cluster_ids, dtype=torch.long))

        # Entity Bias (b_v) - Eq (11)
        if bias_embs is None:
            num_entities = entity_embs.num_embeddings - 1
            self.bias_embs = nn.Embedding(num_entities + 1, 1, padding_idx = 0)
            nn.init.zeros_(self.bias_embs.weight) # Khởi tạo bias bằng 0
        else:
            self.bias_embs = bias_embs

        if temperature is None:
            self.temperature = self.entity_embs.embedding_dim ** 0.5
        else:
            self.temperature = temperature

    def calculate_weights(self, history_relation_ids):
        """
        Thực hiện Eq (13): Tính trọng số W_hu dựa trên tần suất tương tác chiều Tiến.

        Args:
            history_relation_ids: Tensor (Batch, Max_History_Len) - Các relation ID từ user history (đã pad 0).

        Returns:
            weights: Tensor (Batch, L) - Vector trọng số thời gian cho L cụm.
        """
        # 1. Chuẩn bị Tensor để Broadcast (So khớp song song trên GPU)
        # hist_expanded: (Batch, History_Len, 1)
        hist_expanded = history_relation_ids.unsqueeze(-1)

        # clusters_expanded: (1, 1, L) - Chứa [21, 22, 23, 24]
        clusters_expanded = self.cluster_ids.view(1, 1, -1)

        # 2. So khớp và Đếm (Tử số của Eq 13)
        # matches: (Batch, History_Len, L) -> True/1.0 nếu khớp ID cụm thời gian
        matches = (hist_expanded == clusters_expanded).float()

        # counts: (Batch, L) -> Tổng số lần user tương tác vào mỗi cụm
        counts = matches.sum(dim=1)

        # 3. Tính tổng số tương tác thực tế q (Mẫu số của Eq 13)
        # Bằng cách tính tổng của 'counts', ta tự động lờ đi toàn bộ Padding (ID 0)
        # và các relation ID rác (nếu có), vì chúng không match với forward_clusters.
        # q: (Batch, 1)
        q = counts.sum(dim=1, keepdim=True)

        # 4. Chuẩn hóa để ra trọng số (Thêm 1e-9 để tránh lỗi chia cho 0 nếu user chưa có lịch sử)
        # weights: (Batch, L) -> Đảm bảo tổng các phần tử trên mỗi hàng luôn = 1.0 (hoặc 0)
        weights = counts / (q + 1e-9)

        return weights

    def forward(self, user_ids, item_ids, history_relation_ids):
        """
        Tính Reward Score g_R(v | u)

        Args:
            user_ids: (Batch,)
            item_ids: (Batch,) - Item đích (v_hat) mà Agent dự đoán/dừng lại
            history_relation_ids: (Batch, History_Len) - Lịch sử relation của user

        Returns:
            scores: (Batch,) - Điểm reward
        """
        # --- BƯỚC 1: Lấy Embeddings cơ bản ---
        u_e = self.user_embs(user_ids)       # (B, Dim) -> e_u

        v_e = self.entity_embs(item_ids)     # (B, Dim) -> e_v
        v_b = self.bias_embs(item_ids).squeeze(-1) # (B,) -> b_v

        # --- BƯỚC 2: Tính Personalized Interaction Relation (Eq 12) ---
        # r_vu^T = W_hu * V_U

        # a. Tính weights (B, L)
        weights = self.calculate_weights(history_relation_ids)

        # b. Lấy embedding của các cluster V_U^1...L
        # cluster_embs shape: (L, Dim)
        cluster_embs = self.relation_embs(self.cluster_ids)

        # c. Tính tổng có trọng số
        # (B, L) x (L, Dim) -> (B, Dim)
        r_interaction = torch.matmul(weights, cluster_embs)

        # --- BƯỚC 3: Tính Score (Eq 11 & Final Eq) ---
        # g = (e_u + r_interaction) . e_v + b_v

        # Dot product: (e_u + r) * e_v
        query_vector = u_e + r_interaction # (B, Dim)

        # dot_product = torch.sum(query_vector * v_e, dim=1) # (B,)
        # scores = dot_product + v_b # Cộng bias

        distances = torch.norm(query_vector - v_e , p=2, dim=1) + 1e-8
        scores = torch.exp(-(distances - 1.5)) + v_b   # 1.5 is the mean error in TRANSE

        # 1. Scale Score (Chia cho nhiệt độ)
        scaled_score = scores / self.temperature

        # 2. Áp dụng Sigmoid
        rewards = torch.sigmoid(scaled_score)

        return rewards

    # def forward(self, user_ids, item_ids, history_relation_ids):
    #     """
    #     Tính Reward Score g_R(v | u) chuẩn theo Công thức (10)
    #     """
    #     # --- BƯỚC 1: Lấy Embeddings cơ bản ---
    #     u_e = self.user_embs(user_ids)       # (B, Dim) -> e_u
    #     v_e = self.entity_embs(item_ids)     # (B, Dim) -> e_v
    #     v_b = self.bias_embs(item_ids).squeeze(-1) # (B,) -> b_v

    #     # --- BƯỚC 2: Tính Personalized Interaction Relation (Eq 12) ---
    #     weights = self.calculate_weights(history_relation_ids)
    #     cluster_embs = self.relation_embs(self.cluster_ids)
    #     r_interaction = torch.matmul(weights, cluster_embs)

    #     # --- BƯỚC 3: Tính Tử số g_R(e_K | u) (Điểm của item đích) ---
    #     query_vector = u_e + r_interaction # (B, Dim)
    #     numerator_scores = torch.sum(query_vector * v_e, dim=1) + v_b # (B,)

    #     # (Tùy chọn) Scale nhiệt độ nếu bạn muốn duy trì
    #     numerator_scores = numerator_scores / self.temperature

    #     # =====================================================================
    #     # BƯỚC 4: Tính Mẫu số max_{v in V} g_R(v | u) (Công thức 10)
    #     # =====================================================================
    #     # a. Lấy toàn bộ trọng số của tập Entity và Bias
    #     all_entity_embs = self.entity_embs.weight # (Num_Entities, Dim)
    #     all_bias = self.bias_embs.weight.squeeze(-1) # (Num_Entities,)

    #     # b. Nhân vô hướng Query Vector với TOÀN BỘ Entity
    #     # (Batch, Dim) x (Dim, Num_Entities) -> (Batch, Num_Entities)
    #     all_scores = torch.matmul(query_vector, all_entity_embs.transpose(0, 1)) + all_bias

    #     # c. Áp dụng chung mức nhiệt độ (để đồng bộ với tử số)
    #     all_scores = all_scores / self.temperature

    #     # d. Tìm điểm số Max trong toàn tập cho từng User
    #     max_scores, _ = torch.max(all_scores, dim=-1) # (Batch,)

    #     # =====================================================================
    #     # BƯỚC 5: Chốt Reward R_K
    #     # =====================================================================
    #     # Cộng thêm epsilon 1e-8 để chống lỗi chia cho 0
    #     rewards = numerator_scores / (max_scores + 1e-8)

    #     # 🚨 BẢO HIỂM RL (Rất quan trọng):
    #     # Vì phép nhân vô hướng có thể ra số ÂM, nếu tử số âm và mẫu số âm
    #     # thì chia nhau nó lại thành số DƯƠNG (Agent sẽ bị lừa học sai).
    #     # Ta dùng hàm kẹp clamp để ép reward thấp nhất là 0.0, cao nhất là 1.0
    #     rewards = torch.clamp(rewards, min=0.0, max=1.0)

        return rewards


#### TPRecEnvironment

In [43]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

class TPRecEnvironment(nn.Module):
    def __init__(self, tckg, entity_embeddings, relation_embeddings, reward_function, max_path_len, history_len, item_start_id, item_end_id):
        """
        Môi trường TPRec được thiết kế chuẩn xác theo kiến trúc Sequence State
        """
        super(TPRecEnvironment, self).__init__()
        self.tckg = tckg
        self.entity_embs = entity_embeddings
        self.relation_embs = relation_embeddings
        self.max_path_len = max_path_len
        self.history_len = history_len
        self.item_start_id = item_start_id
        self.item_end_id = item_end_id

        # LƯU REWARD FUNCTION ĐƯỢC TRUYỀN VÀO
        self.reward_function = reward_function

        # State tracking
        self.current_entities = None
        self.current_users = None
        self.path_history = None
        self.step_counter = 0

    def reset(self, user_ids):
        """
        Khởi tạo trạng thái s_0 = (u, u, ∅)
        Cập nhật: Nạp thêm target_items để Trọng tài (Môi trường) cầm sẵn đáp án
        """
        batch_size = user_ids.size(0)

        self.current_users = user_ids
        self.current_entities = user_ids

        # History h_k: store (entity, relation) theo chuẩn paper
        self.path_history = torch.zeros((batch_size, self.history_len * 2),     # history_len = k' in paper
                                        dtype=torch.long,
                                        device=user_ids.device)

        self.step_counter = 0
        return self._get_state_embedding()

    def _get_state_embedding(self):
        """
        Kết hợp u, h_k, e_k thành một chuỗi duy nhất đưa vào BLSTM
        """
        # 1. Thêm chiều thời gian (unsqueeze) cho u và e_k
        u_emb = self.entity_embs(self.current_users).unsqueeze(1)    # Shape: (B, 1, d)
        e_emb = self.entity_embs(self.current_entities).unsqueeze(1) # Shape: (B, 1, d)

        # 2. Lấy lịch sử e và r
        e_indices = self.path_history[:, 0::2]
        r_indices = self.path_history[:, 1::2]

        e_vecs = self.entity_embs(e_indices)   # Shape: (B, L, d)
        r_vecs = self.relation_embs(r_indices) # Shape: (B, L, d)

        # 3. Trộn xen kẽ e và r thành chuỗi lịch sử [e1, r2, e2, r3]
        B, L, d = e_vecs.shape
        history_seq = torch.zeros((B, L * 2, d), device=e_vecs.device)
        history_seq[:, 0::2, :] = e_vecs
        history_seq[:, 1::2, :] = r_vecs

        # 4. KẾT NỐI TOÀN BỘ THEO ĐÚNG CÔNG THỨC S_K TRONG PAPER
        full_state_seq = torch.cat([u_emb, history_seq, e_emb], dim=1) # Shape: (B, Seq_Len, d)

        # Trả về ĐÚNG 1 BIẾN DUY NHẤT đại diện cho S_k
        return full_state_seq

    def get_pruned_actions(self, epsilon=150):   # used by def get_action_space_batch
        """
        Phiên bản Tối ưu hóa Vectorization (GPU Accelerated)
        """
        batch_size = self.current_users.size(0)
        device = self.current_users.device

        # 1. KÉO VỀ CPU MỘT LẦN DUY NHẤT: Tránh gọi .item() N lần
        curr_nodes_cpu = self.current_entities.tolist()

        # Tra cứu láng giềng cực nhanh trên CPU bằng List Comprehension
        batch_neighbors = [self.tckg.get_neighbors(node) for node in curr_nodes_cpu]


        # Tìm node có số lượng láng giềng lớn nhất trong batch này
        lengths = [len(n) for n in batch_neighbors]
        max_len = max(lengths) if lengths else 0

        valid_actions = []

        # Nếu toàn bộ batch đều rơi vào ngõ cụt
        if max_len == 0:
            return [[] for _ in range(batch_size)]

        # 2. TẠO MA TRẬN PADDING TRÊN CPU
        batch_rels = torch.zeros((batch_size, max_len), dtype=torch.long)
        batch_next_nodes = torch.zeros((batch_size, max_len), dtype=torch.long)
        mask = torch.zeros((batch_size, max_len), dtype=torch.bool)

        # Điền dữ liệu vào ma trận
        for i, neighbors in enumerate(batch_neighbors):
            num_n = lengths[i]
            if num_n > 0:
                batch_rels[i, :num_n] = torch.tensor([n[0] for n in neighbors])            #neighbor: head -> (r, t) rel use 0, node use 1
                batch_next_nodes[i, :num_n] = torch.tensor([n[1] for n in neighbors])
                mask[i, :num_n] = True

        # 3. ĐẨY TOÀN BỘ MA TRẬN LÊN GPU MỘT LẦN DUY NHẤT
        batch_rels = batch_rels.to(device)
        batch_next_nodes = batch_next_nodes.to(device)
        mask = mask.to(device)

        # 4. TÍNH TOÁN SONG SONG BẰNG MA TRẬN TRÊN GPU
        curr_emb = self.entity_embs(self.current_entities) # Lấy node HIỆN TẠI (Đã sửa)
        r_emb = self.relation_embs(batch_rels)
        n_emb = self.entity_embs(batch_next_nodes)

        # Tính Query: Cần unsqueeze curr_emb để cộng broadcast với r_emb
        query = curr_emb.unsqueeze(1) + r_emb

        # LỖI CŨ CẦN XÓA: scores = torch.sum(query * n_emb, dim=-1)

        # CÁCH CHUẨN MỚI DÀNH CHO TRANSE:
        # Tính khoảng cách L2 (Euclidean distance) giữa (h+r) và t
        # Khoảng cách càng nhỏ càng tốt -> Thêm dấu trừ (-) để hàm topk chọn giá trị gần 0 nhất
        distances = torch.norm(query - n_emb, p=2, dim=-1)
        scores = torch.exp(-(distances - 1.5))

        # CHE PADDING (Masking)
        scores = scores.masked_fill(~mask, float('-inf'))

        # 5. CHỌN TOP-K CHO CẢ BATCH
        k = min(epsilon, max_len)
        top_scores, top_indices = torch.topk(scores, k, dim=1) # (B, k)

        # 6. KÉO KẾT QUẢ VỀ LẠI CPU
        top_indices_cpu = top_indices.tolist()
        batch_rels_cpu = batch_rels.tolist()
        batch_nodes_cpu = batch_next_nodes.tolist()

        # Giải nén thành dạng List gốc
        for i in range(batch_size):
            actions_i = []
            num_real = lengths[i]

            if num_real > 0:
                valid_k = min(k, num_real)
                for j in range(valid_k):
                    idx = top_indices_cpu[i][j]
                    actions_i.append((batch_rels_cpu[i][idx], batch_nodes_cpu[i][idx]))

            valid_actions.append(actions_i)

        return valid_actions

    def get_action_space_batch(self):
        """
        Lấy không gian hành động cho cả Batch và Padding thành Tensor.
        Tích hợp LUẬT CHỐNG LẶP VÒNG (Cycle Prevention) cực mượt.
        """
        raw_actions_list = self.get_pruned_actions()
        batch_size = len(raw_actions_list)

        lengths = [len(acts) for acts in raw_actions_list]
        max_len = max(lengths) if lengths else 0
        if max_len == 0:
            max_len = 1

        device = self.current_entities.device

        r_indices = torch.zeros((batch_size, max_len), dtype=torch.long, device=device)
        e_indices = torch.zeros((batch_size, max_len), dtype=torch.long, device=device)
        action_mask = torch.zeros((batch_size, max_len), dtype=torch.float, device=device)

        # [MỚI] Lấy lịch sử các Entity đã đi qua (Chỉ lấy index chẵn trong path_history)
        if self.path_history is not None:
            visited_entities = self.path_history[:, 0::2].tolist()
        else:
            visited_entities = [[] for _ in range(batch_size)]

        for i, actions in enumerate(raw_actions_list): # i là user thứ i trong batch
            num_acts = len(actions)
            if num_acts > 0:
                # =====================================================================
                # 1. TẠO DANH SÁCH ĐEN (BLACKLIST) CHO USER i
                # =====================================================================
                blacklist = set(visited_entities[i])            # Các node trong lịch sử
                blacklist.add(self.current_users[i].item())     # Cấm quay về User gốc
                blacklist.add(self.current_entities[i].item())  # Cấm dẫm tại chỗ (Self-loop)

                rs = []
                es = []
                valid_masks = []

                # =====================================================================
                # 2. KIỂM TRA TỪNG HÀNH ĐỘNG VỚI BLACKLIST
                # =====================================================================
                for rel, target_node in actions:
                    rs.append(rel)
                    es.append(target_node)

                    # Nếu target_node nằm trong blacklist -> Mask = 0.0 (Cấm cửa)
                    # Nếu là node mới toanh -> Mask = 1.0 (Cho phép)
                    if target_node in blacklist:
                        valid_masks.append(0.0)
                    else:
                        valid_masks.append(1.0)

                # =====================================================================
                # 3. [QUAN TRỌNG] BẢO HIỂM CHỐNG CRASH (DEAD-END PREVENTION)
                # =====================================================================
                # Xử lý trường hợp Agent đi vào ngõ cụt (Tất cả các ngã rẽ đều đã đi qua)
                # Nếu sum == 0, toàn bộ Mask là 0 -> Điểm -vô cực -> Hàm Softmax sẽ trả về NaN!
                if sum(valid_masks) == 0.0:
                    valid_masks[0] = 1.0 # Bắt buộc mở khóa 1 đường bất kỳ để mạng nơ-ron không bị nổ (Crash)

                # 4. Gán dữ liệu vào Tensor
                r_indices[i, :num_acts] = torch.tensor(rs, device=device)
                e_indices[i, :num_acts] = torch.tensor(es, device=device)
                action_mask[i, :num_acts] = torch.tensor(valid_masks, dtype=torch.float, device=device)

        # 5. Lấy vector nhúng từ ID
        r_emb = self.relation_embs(r_indices)
        e_emb = self.entity_embs(e_indices)

        # CHỐT: Ghép theo đúng định nghĩa Toán học Action = (Relation, Entity)
        action_embs = torch.cat([r_emb, e_emb], dim=-1)

        return action_embs, action_mask, raw_actions_list

    def step(self, actions):    # used  by def step_with_indices
        """
        Transition function (Eq 9): Chuyển trạng thái sang bước k+1
        """
        device = self.current_entities.device
        batch_size = len(actions)

        rels_list = [a[0] for a in actions]
        ents_list = [a[1] for a in actions]

        next_relations = torch.tensor(rels_list, dtype=torch.long, device=device)
        next_entities = torch.tensor(ents_list, dtype=torch.long, device=device)

        # Lấy Node hiện tại làm điểm xuất phát (e_{k-1})
        curr_e = self.current_entities.unsqueeze(1)
        new_r = next_relations.unsqueeze(1)

        # Nối lại theo chuẩn Paper: [e_{k-1}, r_k]
        new_entry = torch.cat([curr_e, new_r], dim=1)

        history_shifted = self.path_history[:, 2:]
        self.path_history = torch.cat([history_shifted, new_entry], dim=1)

        self.current_entities = next_entities #tensor([8673, 8614])
        self.step_counter += 1
        done = (self.step_counter >= self.max_path_len)

        return self._get_state_embedding(), done

    def step_with_indices(self, action_indices, raw_actions_list):
        selected_real_actions = []
        batch_size = len(action_indices)

        for i in range(batch_size):
            idx = action_indices[i].item()

            # [ĐÃ SỬA TÊN] Dùng 'node_acts' thay vì 'user_acts'
            node_acts = raw_actions_list[i]

            if len(node_acts) > 0:
                idx = min(idx, len(node_acts) - 1)
                real_action = node_acts[idx]
            else:
                curr_node = self.current_entities[i].item()
                real_action = (0, curr_node) # Dead-end: Tự trỏ về chính nó

            selected_real_actions.append(real_action)

        return self.step(selected_real_actions)

    def get_reward(self):
        """
        Gọi khi done=True (Hoặc ở các bước lẻ).
        Tính Time-aware Reward g_R(v|u). Nếu không phải Item -> Reward = 0.0

        Lưu ý: tham số truyền vào nên là 1 Tensor chứa TOÀN BỘ các ID của Item có trong Đồ thị.
        """
        user_ids = self.current_users
        item_ids = self.current_entities

        batch_history_relations = []

        for user_id in user_ids.tolist():
            batch_neighbors = self.tckg.get_neighbors(user_id)
            user_rels = [rel for (rel, tail) in batch_neighbors]
            user_rels_tensor = torch.tensor(user_rels, dtype=torch.long)
            batch_history_relations.append(user_rels_tensor)

        history_relation_ids = pad_sequence(
            batch_history_relations,
            batch_first=True,
            padding_value=0
        ).to(self.current_users.device)

        # 1. SOFT REWARD: Tính điểm dẫn đường cho TOÀN BỘ node hiện tại (bất chấp là Item hay Entity)
        # rewards có shape: (Batch_size,)
        rewards = self.reward_function(user_ids, item_ids, history_relation_ids)

        # =====================================================================
        # 2. [MỚI] MASKING: TẮT REWARD VỀ 0.0 NẾU KHÔNG PHẢI LÀ ITEM
        # =====================================================================
        is_item_mask = (item_ids >= self.item_start_id) & (item_ids <= self.item_end_id)

        rewards = torch.where(is_item_mask, rewards, torch.zeros_like(rewards))

        return rewards

#### TPRecPolicy

In [44]:
import torch
import torch.nn as nn

class TPRecPolicy(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout=0.1):
        super(TPRecPolicy, self).__init__()

        # BLSTM giờ đây chỉ cần nhận input_size = embed_dim (không cần nhân 2 nữa)
        self.blstm = nn.LSTM(input_size=embed_dim,
                             hidden_size=hidden_dim,
                             num_layers=1,
                             batch_first=True,
                             bidirectional=True)

        # W1 giờ đây chỉ cần nhận đầu ra của BLSTM
        self.W1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

        self.Wa = nn.Linear(hidden_dim, embed_dim * 2)
        self.Wc = nn.Linear(hidden_dim, 1)

    def forward(self, full_state_seq, action_embs, action_mask):
        # 1. Đưa toàn bộ s_k vào BLSTM như công thức (15)
        lstm_out, _ = self.blstm(full_state_seq)

        # # Lấy trạng thái ở bước thời gian cuối cùng đại diện cho toàn bộ chuỗirelation_idrelation_id
        # lstm_last = lstm_out[:, -1, :]

        # Vì bidirectional=True, hidden_dim này thực chất là 2 phần ghép lại
        half_dim = self.blstm.hidden_size

        # Lấy trạng thái Tóm tắt Chiều đi tới (Ở index cuối cùng: -1)
        forward_last = lstm_out[:, -1, :half_dim]

        # Lấy trạng thái Tóm tắt Chiều đi lùi (Ở index đầu tiên: 0)
        backward_last = lstm_out[:, 0, half_dim:]

        # Ghép 2 tóm tắt này lại thành một Context Vector hoàn hảo
        lstm_last = torch.cat([forward_last, backward_last], dim=-1)

        # 2. Tính x_k theo đúng phương trình
        x_k = self.dropout(torch.relu(self.W1(lstm_last)))

        # --- (Phần tính Actor và Critic giữ nguyên) ---
        query = self.Wa(x_k).unsqueeze(1)
        scores = torch.sum(query * action_embs, dim=-1)
        scores = scores.masked_fill(action_mask.bool() == False, float('-inf'))
        probs = torch.softmax(scores, dim=-1)

        value_baseline = self.Wc(x_k).squeeze(-1)

        return probs, value_baseline

#### TPRecLightningModel

In [45]:
import pytorch_lightning as pl
import torch
import torch.optim as optim

class TPRecLightningModel(pl.LightningModule):
    def __init__(self, interaction_cluster_ids, env, policy_net, learning_rate):
        super().__init__()

        # Lưu lại hyperparameter (tự động log vào tensorboard nếu dùng)
        self.save_hyperparameters(ignore=['env', 'policy_net'])

        self.interaction_cluster_ids = interaction_cluster_ids
        self.env = env
        self.policy_net = policy_net
        self.learning_rate = learning_rate

    def forward(self, full_state_seq, action_embs, action_mask):
        """
        Đạo diễn Lightning chuyển tiếp ĐÚNG 3 tham số cho Diễn viên Policy Network
        """
        probs, value_baseline =  self.policy_net(full_state_seq, action_embs, action_mask)
        return probs, value_baseline

    def training_step(self, batch, batch_idx):
        """
        Phiên bản Thử nghiệm: Thưởng dọc đường (Multi-hop) nhưng BỎ QUA Bước 1.
        Agent chỉ được nhận Reward khi đến các trạm Item ở t=2, t=4...
        """
        batch_users, _ = batch[0], batch[1]

        batch_size = batch_users.size(0)
        full_state_seq = self.env.reset(batch_users)

        saved_log_probs = []
        saved_values = []
        saved_rewards = [] # Mảng hứng phần thưởng từng bước

        for t in range(self.env.max_path_len):
            action_embs, action_mask, raw_actions = self.env.get_action_space_batch()

            probs, value_baseline = self(full_state_seq, action_embs, action_mask)

            m = torch.distributions.Categorical(probs)
            action_indices = m.sample()

            saved_log_probs.append(m.log_prob(action_indices))
            saved_values.append(value_baseline)

            # Thực hiện bước đi
            full_state_seq, done = self.env.step_with_indices(action_indices, raw_actions)

            # # =====================================================================
            # # [CỐT LÕI] BỘ LỌC THỜI GIAN CHO PHẦN THƯỞNG
            # # =====================================================================
            # # Mặc định phần thưởng của bước này là 0.0 cho toàn bộ Batch
            # step_reward = torch.zeros(batch_size, device=self.device)

            # # CHỈ cho phép gọi get_reward() tính điểm nếu:
            # # 1. Không phải bước đầu tiên (t > 0) -> Bỏ qua thiên vị Item cũ
            # # 2. Là bước chẵn (t % 2 == 0) -> Đảm bảo đang đứng ở tầng Item (Hop 3, Hop 5...)
            # if t > 0 and t % 2 == 0:
            #     # Hàm get_reward() đã có sẵn mask kiểm tra Entity/Item bên trong nó
            #     step_reward = self.env.get_reward().detach()

            # saved_rewards.append(step_reward)

        # =====================================================================
        # TÍNH TOÁN LỢI TỨC G THEO ĐỊNH LÝ BELLMAN
        # =====================================================================
        step_reward = self.env.get_reward().detach() ### theo paper thì chỉ tính reward cho terminate state
        saved_rewards.append(step_reward)

        gamma = 0.99
        returns = []

        # G khởi tạo bằng 0
        G = torch.zeros_like(saved_rewards[0])

        # Duyệt ngược từ bước cuối về bước đầu
        for R_step in reversed(saved_rewards):
            G = R_step + gamma * G
            returns.insert(0, G)

        policy_loss = 0
        value_loss = 0

        # CÔNG THỨC 18: Tính Loss REINFORCE
        for G_val, log_prob, value_baseline in zip(returns, saved_log_probs, saved_values):
            advantage = G_val - value_baseline.detach()

            step_policy_loss = -log_prob * advantage
            policy_loss += step_policy_loss.mean()

            step_value_loss = torch.nn.functional.mse_loss(value_baseline, G_val)
            value_loss += step_value_loss

        total_loss = policy_loss + value_loss

        # Ghi nhận log: Tính tổng điểm thực tế gom được trong Episode
        total_episode_reward = sum(saved_rewards)
        self.log('train_reward', total_episode_reward.mean(), prog_bar=True, on_step=False, on_epoch=True)
        self.log('train_loss', total_loss, prog_bar=False, on_step=False, on_epoch=True)

        return total_loss

    def validation_step(self, batch, batch_idx):
        """
        Vòng lặp Đánh giá (Validation) mô phỏng chính xác Section 3.4 của TPRec.
        Bao gồm: Val Loss, Val Reward, và Re-ranking bằng Công thức (20) để tính HR/NDCG.
        """
        batch_users = batch[0]
        target_items = batch[1]
        W_T_hat = batch[2] # Shape: (Batch_size, N_Clusters)

        batch_size = batch_users.size(0)

        full_state_seq = self.env.reset(batch_users)

        saved_log_probs = []
        saved_values = []
        saved_rewards = [] # Mảng hứng phần thưởng từng bước

        # =====================================================================
        # 1. PHASE 1: AGENT TÌM KIẾM ĐƯỜNG ĐI (Thu thập tập ứng viên V_hat)
        # =====================================================================
        for t in range(self.env.max_path_len):
            action_embs, action_mask, raw_actions = self.env.get_action_space_batch()
            probs, value_baseline = self(full_state_seq, action_embs, action_mask)

            # Ở chế độ Val, relation_embeingsta dùng argmax (Greedy) để chọn đường đi tự tin nhất
            action_indices = torch.argmax(probs, dim=-1)

            m = torch.distributions.Categorical(probs)
            saved_log_probs.append(m.log_prob(action_indices))
            saved_values.append(value_baseline)

            full_state_seq, done = self.env.step_with_indices(action_indices, raw_actions)

            # =====================================================================
            # [CỐT LÕI] BỘ LỌC THỜI GIAN CHO PHẦN THƯỞNG
            # =====================================================================
            # Mặc định phần thưởng của bước này là 0.0 cho toàn bộ Batch
            step_reward = torch.zeros(batch_size, device=self.device)

            # CHỈ cho phép gọi get_reward() tính điểm nếu:
            # 1. Không phải bước đầu tiên (t > 0) -> Bỏ qua thiên vị Item cũ
            # 2. Là bước chẵn (t % 2 == 0) -> Đảm bảo đang đứng ở tầng Item (Hop 3, Hop 5...)
            if t > 0 and t % 2 == 0:
                # Hàm get_reward() đã có sẵn mask kiểm tra Entity/Item bên trong nó
                step_reward = self.env.get_reward().detach()

            saved_rewards.append(step_reward)

        # =====================================================================
        # 2. PHASE 2: RE-RANKING BẰNG CÔNG THỨC (20) VÀ TÍNH HR/NDCG
        # =====================================================================
        embed_dim = self.env.entity_embs.embedding_dim
        e_V_hat = action_embs[:, :, embed_dim:] # Shape: (Batch, Max_Len, Embed_Dim)

        e_u = self.env.entity_embs(batch_users) # Shape: (Batch, Embed_Dim)

        # =====================================================================
        # [MỚI] ÁP DỤNG CÔNG THỨC 19: Tính Temporal Relation Vector (r_W_T)
        # =====================================================================
        # Thực hiện phép nhân ma trận (Matrix Multiplication): W_T_hat * V_U_T
        # W_T_hat shape            : (Batch_size, N_Clusters)
        # V_U_T : (N_Clusters, Embed_Dim)
        cluster_ids_tensor = torch.tensor(
            self.interaction_cluster_ids,
            dtype=torch.long,
            device=self.device # Đảm bảo Tensor nằm cùng chỗ với Model
        )

        # 2. Truyền Tensor vào lớp Embedding
        V_U_T = self.env.relation_embs(cluster_ids_tensor)
        r_W_T = torch.matmul(W_T_hat.float(), V_U_T) # Shape trả về: (Batch_size, Embed_Dim)

        # Tạo Query Vector theo Công thức 20: (e_u + r_W_T)
        query_vector = e_u + r_W_T # Shape: (Batch, Embed_Dim)

        # Áp dụng Công thức (20): Phép nhân vô hướng (Dot Product) với tập ứng viên e_V_hat
        scores = torch.sum(query_vector.unsqueeze(1) * e_V_hat, dim=-1) # Shape: (Batch, Max_Len)

        # Masking: Ép điểm các node đệm (padding) hoặc node cấm về Âm vô cực
        scores = scores.masked_fill(action_mask == 0, float('-inf'))

        # Sort descending để lấy Top 10 (Chính là hàm Sort() trong Eq 20)
        k = min(10, scores.size(1))
        _, top10_indices = torch.topk(scores, k=k, dim=1)

        # Tính toán Hit Ratio và NDCG
        hits = 0.0
        ndcg = 0.0

        for i in range(batch_size):
            user_acts = raw_actions[i]
            target = target_items[i].item()

            top10_ids = []
            for idx in top10_indices[i].tolist():
                if idx < len(user_acts):
                    top10_ids.append(user_acts[idx][1]) # Index 1 là Entity ID (Item)

            if target in top10_ids:
                hits += 1.0
                rank = top10_ids.index(target) + 1
                ndcg += 1.0 / np.log2(rank + 1)

        val_hr = hits / batch_size
        val_ndcg = ndcg / batch_size

        # =====================================================================
        # 3. PHASE 3: TÍNH VAL LOSS VÀ VAL REWARD
        # =====================================================================
        # =====================================================================
        # TÍNH TOÁN LỢI TỨC G THEO ĐỊNH LÝ BELLMAN
        # =====================================================================
        gamma = 0.99
        returns = []

        # G khởi tạo bằng 0
        G = torch.zeros_like(saved_rewards[0])

        # Duyệt ngược từ bước cuối về bước đầu
        for R_step in reversed(saved_rewards):
            G = R_step + gamma * G
            returns.insert(0, G)

        policy_loss = 0
        value_loss = 0

        # CÔNG THỨC 18: Tính Loss REINFORCE
        for G_val, log_prob, value_baseline in zip(returns, saved_log_probs, saved_values):
            advantage = G_val - value_baseline.detach()

            step_policy_loss = -log_prob * advantage
            policy_loss += step_policy_loss.mean()

            step_value_loss = torch.nn.functional.mse_loss(value_baseline, G_val)
            value_loss += step_value_loss

        total_loss = policy_loss + value_loss

        # Ghi nhận log: Tính tổng điểm thực tế gom được trong Episode
        total_episode_reward = sum(saved_rewards)

        # 4. GHI LOG LÊN TENSORBOARD
        self.log('val_reward', total_episode_reward.mean(), prog_bar=True, on_epoch=True)
        self.log('val_loss', total_loss, prog_bar=True, on_epoch=True)
        self.log('val_hr@10', val_hr, prog_bar=True, on_epoch=True)
        self.log('val_ndcg@10', val_ndcg, prog_bar=True, on_epoch=True)

        return total_loss

    def configure_optimizers(self):
        # Lọc ra CHỈ những tham số đang requires_grad=True (Mạng Policy)
        # Giúp tránh lỗi tối ưu hóa các tham số đã bị freeze (như Embeddings)
        trainable_params = filter(lambda p: p.requires_grad, self.parameters())

        optimizer = optim.Adam(trainable_params, lr=self.learning_rate)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)

        return [optimizer], [scheduler]

### Load Dataset

In [46]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class InteractionDataset(Dataset):
    def __init__(self, df, n_clusters):
        """Nhận vào một DataFrame và chuyển đổi các cột cần thiết thành mảng NumPy"""
        self.users = df['user_id'].values
        self.entities = df['entity_id'].values

        prob_cols = [f'prob_cluster_{i}' for i in range(n_clusters)]
        self.w_t_hat_matrix = df[prob_cols].values

    def __len__(self):
        """Trả về tổng số dòng dữ liệu"""
        return len(self.users)

    def __getitem__(self, idx):
        """Lấy dữ liệu tại vị trí idx và biến thành PyTorch Tensor"""
        user_tensor = torch.tensor(self.users[idx], dtype=torch.long)
        entity_tensor = torch.tensor(self.entities[idx], dtype=torch.long)
        w_t_hat_tensor = torch.tensor(self.w_t_hat_matrix[idx], dtype=torch.float32)

        return user_tensor, entity_tensor, w_t_hat_tensor

# 1. Đọc file CSV (Lưu ý: Dùng đúng file CSV đã được nối cột prob_cluster_X)
train_df = pd.read_csv(f'/content/data/{NAME}/{NAME}_gmm_train_interactions.csv')
val_df = pd.read_csv(f'/content/data/{NAME}/{NAME}_gmm_val_interactions.csv')
# test_df = pd.read_csv(f'./data/{NAME}/{NAME}_gmm_test_interactions.csv')

# 2. Bọc DataFrame vào class Dataset (Nhớ truyền tham số n_clusters)
train_dataset = InteractionDataset(train_df, n_clusters=N_CLUSTERS)
val_dataset = InteractionDataset(val_df, n_clusters=N_CLUSTERS)
# test_dataset = InteractionDataset(test_df, n_clusters=N_CLUSTERS)

# 3. Đưa Dataset vào DataLoader
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=8)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=8)
# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)


### Main function

In [47]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

cfg = Config()
print("🔥 Starting TPRec Training with PyTorch Lightning...")

# -------------------------------------------------
# BƯỚC 1: Xây dựng Đồ thị Tri thức (TCKG)
# -------------------------------------------------
# Lưu ý: Cần đảm bảo file CSV đã được map ID về dạng số nguyên liên tục (0, 1, 2...)
print("Loading Knowledge Graph...")
# Tính offset tự động (như class TCKG tối ưu tôi đã viết)
tckg = TCKG(cfg.TCKG_PATH)

# -------------------------------------------------
# BƯỚC 2: Khởi tạo Embeddings từ file Pickle
# -------------------------------------------------
print("Loading Pre-trained TransE Embeddings...")
pickle_file_path = f'/content/pickle/book_transE_embeddings_2026-03-01_12-11-01.pkl'

with open(pickle_file_path, 'rb') as f:
    saved_data = pickle.load(f)

pretrained_ent = saved_data['entity_embeddings']
pretrained_rel = saved_data['relation_embeddings']

# 2. Chuyển đổi sang PyTorch Tensor (ép kiểu Float32 để tính toán neural network)
# Nếu data đang là Numpy array:
if isinstance(pretrained_ent, np.ndarray):
    ent_tensor = torch.tensor(pretrained_ent, dtype=torch.float32)
    rel_tensor = torch.tensor(pretrained_rel, dtype=torch.float32)
else:
    # Nếu data đã là Tensor sẵn:
    ent_tensor = pretrained_ent.clone().detach().float()
    rel_tensor = pretrained_rel.clone().detach().float()

# 3. Nạp vào nn.Embedding
# freeze=False: Cho phép RL Agent tiếp tục cập nhật (fine-tune) vector trong lúc tìm đường
# freeze=True: Khóa cứng vector, RL Agent chỉ học Policy Network (Học nhanh hơn, chống overfit)
entity_embs = nn.Embedding.from_pretrained(ent_tensor, freeze=False, padding_idx=0)
relation_embs = nn.Embedding.from_pretrained(rel_tensor, freeze=False, padding_idx=0)


# -------------------------------------------------
# BƯỚC 3: Khởi tạo Reward Function
# -------------------------------------------------
print("Setting up Reward Function...")
reward_func = TimeAwareRewardFunction(
    user_embs=entity_embs,    # Chia sẻ trọng số với Env
    entity_embs=entity_embs,
    relation_embs=relation_embs,
    interaction_cluster_ids=cfg.INTERACTION_CLUSTER_IDS,
    bias_embs=None, # Tự tạo bias mới
    temperature= 1
)

# -------------------------------------------------
# BƯỚC 4: Khởi tạo Môi trường (Environment)
# -------------------------------------------------
print("Setting up Environment...")
env = TPRecEnvironment(
    tckg=tckg,
    entity_embeddings=entity_embs,
    relation_embeddings=relation_embs,
    reward_function=reward_func, # Inject reward vào env
    max_path_len=cfg.MAX_PATH_LEN,
    history_len=cfg.HISTORY_LEN,
    item_start_id=cfg.ITEM_START_ID,
    item_end_id=cfg.ITEM_END_ID
)

# -------------------------------------------------
# BƯỚC 5: Khởi tạo Policy Network (Agent)
# -------------------------------------------------
print("Building Policy Network...")
policy_net = TPRecPolicy(
        embed_dim=cfg.EMBED_DIM,      # Ví dụ: 64
        hidden_dim=cfg.HIDDEN_DIM,    # Ví dụ: 128
        dropout=0.1                   # Tỉ lệ dropout giúp chống Overfit (theo paper)
    )

# BƯỚC 6: ĐÓNG GÓI VÀO LIGHTNING MODEL
print("Packing into Lightning Module...")
lightning_model = TPRecLightningModel(
    interaction_cluster_ids=cfg.INTERACTION_CLUSTER_IDS,
    env=env,
    policy_net=policy_net,
    learning_rate=cfg.LEARNING_RATE
)

# BƯỚC 7: CẤU HÌNH LƯU BEST MODEL (CHECKPOINT)
# Tự động theo dõi 'train_reward' ở cuối mỗi epoch và lưu lại bản có điểm cao nhất
checkpoint_callback = ModelCheckpoint(
    dirpath=cfg.SAVE_DIR,
    filename='tprec-best-{epoch:02d}-{train_reward:.4f}',
    monitor='train_reward',
    mode='max', # Lưu model có reward lớn nhất
    save_top_k=1,
    save_last=True # Lưu thêm model ở epoch cuối cùng để phòng hờ
)

# 2. THÊM CALLBACK EARLY STOPPING VÀO ĐÂY
early_stop_callback = EarlyStopping(
      monitor='train_reward',   # Phải cùng tên với biến monitor ở Checkpoint
      min_delta=0.001,       # Sự thay đổi tối thiểu để được tính là "có cải thiện"
      patience=500,           # Sức chịu đựng: Cho phép mô hình dậm chân tại chỗ tối đa 10 Epoch
      verbose=True,          # Bật in thông báo ra màn hình khi Early Stop kích hoạt
      mode='max'             # 'max' vì ta muốn chỉ số HR@10/Reward càng lớn càng tốt
  )

# BƯỚC 8: KHỞI TẠO TRAINER VÀ BẮT ĐẦU CHẠY
print("Initializing Lightning Trainer...")
trainer = pl.Trainer(
    max_epochs=cfg.NUM_EPOCHS,
    accelerator="auto", # Tự động tìm và dùng GPU nếu có
    devices=1,
    gradient_clip_val=1.0, # Tự động áp dụng Gradient Clipping
    callbacks=[checkpoint_callback, early_stop_callback],
    enable_progress_bar=True,
    # log_every_n_steps=10,
    num_sanity_val_steps=0
)

print("🚀 Bắt đầu huấn luyện...")
# DataLoader của bạn cần được truyền vào đây
trainer.fit(
        model=lightning_model,
        train_dataloaders=train_loader
    )


🔥 Starting TPRec Training with PyTorch Lightning...
Loading Knowledge Graph...
Loading TCKG from ./data/book/book_TCKG.csv...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


TCKG Loaded successfully. Graph construction complete.
Loading Pre-trained TransE Embeddings...
Setting up Reward Function...
Setting up Environment...
Building Policy Network...
Packing into Lightning Module...
Initializing Lightning Trainer...
🚀 Bắt đầu huấn luyện...


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ env        │ TPRecEnvironment │  2.5 M │ train │     0 │
│ 1 │ policy_net │ TPRecPolicy      │  823 K │ train │     0 │
└───┴────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 3.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.4 M                                                                                                
Total estimated model params size (MB): 13                                                                         
Modules in train mode: 11                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.callbacks.early_stopping:Metric train_reward improved. New best score: 0.947
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_reward improved by 0.006 >= min_delta = 0.001. New best score: 0.953
INFO:pytorch_lightning.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 465, in advance
    self.epoch_loop.run(self._data_fetcher)
  File

TypeError: object of type 'NoneType' has no len()

In [ ]:
# trainer.validate(dataloaders=test_loader, ckpt_path='best')